In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy


In [3]:

DOMAIN_SET = ['angry', 'childlike', 'depressed', 'neutral', 'old', 'proud', 'strutting']
DATA_DIR = './data/ActionStyleDataset/'
DATASET_DETAILS = {
    'prefix': 'ActionStyle-',
    'suffix': '-clip.mat',
    'resnet_feature': 'clip_features',
    'split_file_name': 'instanceSplit_actionStyle_unseen2.mat',
}
NUM_LABELS = 5

In [4]:
import sys

sys.argv.extend([
    "--encoder_layer_sizes", "512", "512",
    "--decoder_layer_sizes", "512", "512",
])

In [5]:
result = {}

## Base

In [6]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [7]:
result["Base"] = run_all_senario(main_base, DOMAIN_SET, input_dim=512, num_trial=6)

angry -> childlike
per-class acc:0.7120, seen acc:0.9375, unseen acc:0.4865, H:0.6406
per-class acc:0.5122, seen acc:0.9375, unseen acc:0.0870, H:0.1592
per-class acc:0.5435, seen acc:1.0000, unseen acc:0.0870, H:0.1600
per-class acc:0.7432, seen acc:0.8125, unseen acc:0.6739, H:0.7367
per-class acc:0.7016, seen acc:0.9167, unseen acc:0.4865, H:0.6356
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     93.40 ± 2.82
Unseen:   30.35 ± 11.40
H-mean:   38.87 ± 12.93
angry -> depressed
per-class acc:0.5103, seen acc:0.9231, unseen acc:0.0976, H:0.1765
per-class acc:0.5598, seen acc:1.0000, unseen acc:0.1196, H:0.2136
per-class acc:0.6413, seen acc:1.0000, unseen acc:0.2826, H:0.4407
per-class acc:0.5997, seen acc:0.9091, unseen acc:0.2903, H:0.4401
per-class acc:0.4855, seen acc:0.8125, unseen acc:0.1585, H:0.2653
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     94.08 ± 3.07
Unseen:   15.81 ± 4.59
H-mean:   25.60 ± 6.87
angry -> neutral
p

# GZSDA

In [8]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [9]:
result["GZSDA"] = run_all_senario(main_gzsda, DOMAIN_SET, input_dim=512, num_trial=6)

angry -> childlike
per-class acc:0.7120, seen acc:0.9375, unseen acc:0.4865, H:0.6406
per-class acc:0.4688, seen acc:0.9375, unseen acc:0.0000, H:0.0000
per-class acc:0.5109, seen acc:1.0000, unseen acc:0.0217, H:0.0426
per-class acc:0.7106, seen acc:0.8125, unseen acc:0.6087, H:0.6960
per-class acc:0.6182, seen acc:0.7500, unseen acc:0.4865, H:0.5902
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     90.62 ± 4.19
Unseen:   26.72 ± 11.77
H-mean:   32.82 ± 14.12
angry -> depressed
per-class acc:0.4737, seen acc:0.9231, unseen acc:0.0244, H:0.0475
per-class acc:0.5272, seen acc:1.0000, unseen acc:0.0543, H:0.1031
per-class acc:0.5761, seen acc:1.0000, unseen acc:0.1522, H:0.2642
per-class acc:0.5610, seen acc:0.9091, unseen acc:0.2129, H:0.3450
per-class acc:0.3742, seen acc:0.6875, unseen acc:0.0610, H:0.1120
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     91.99 ± 4.95
Unseen:   8.41 ± 3.33
H-mean:   14.53 ± 5.40
angry -> neutral
pe

## m0

In [10]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [11]:
result["Our"] = run_all_senario(main_m0, DOMAIN_SET, input_dim=512, num_trial=6)

angry -> childlike
per-class acc:0.6771, seen acc:0.8542, unseen acc:0.5000, H:0.6308
per-class acc:0.6658, seen acc:0.8750, unseen acc:0.4565, H:0.6000
per-class acc:0.7065, seen acc:1.0000, unseen acc:0.4130, H:0.5846
per-class acc:0.8736, seen acc:0.8125, unseen acc:0.9348, H:0.8694
per-class acc:0.5833, seen acc:0.6667, unseen acc:0.5000, H:0.5714
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     86.81 ± 5.12
Unseen:   46.74 ± 12.14
H-mean:   54.27 ± 11.76
angry -> depressed
per-class acc:0.6932, seen acc:0.9231, unseen acc:0.4634, H:0.6171
per-class acc:0.6315, seen acc:0.8500, unseen acc:0.4130, H:0.5559
per-class acc:0.7065, seen acc:1.0000, unseen acc:0.4130, H:0.5846
per-class acc:0.8755, seen acc:0.8636, unseen acc:0.8873, H:0.8753
per-class acc:0.5329, seen acc:0.5625, unseen acc:0.5034, H:0.5313
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     86.65 ± 6.62
Unseen:   44.67 ± 11.54
H-mean:   52.74 ± 11.71
angry -> neutral

## m1: seperate after encoder

In [12]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [13]:
result["Our+GRE"] = run_all_senario(main_m1, DOMAIN_SET, input_dim=512, num_trial=6)

angry -> childlike
per-class acc:0.7188, seen acc:0.9375, unseen acc:0.5000, H:0.6522
per-class acc:0.6671, seen acc:0.8125, unseen acc:0.5217, H:0.6354
per-class acc:0.7065, seen acc:1.0000, unseen acc:0.4130, H:0.5846
per-class acc:0.8845, seen acc:0.8125, unseen acc:0.9565, H:0.8786
per-class acc:0.5417, seen acc:0.5833, unseen acc:0.5000, H:0.5385
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     85.76 ± 6.49
Unseen:   48.19 ± 12.44
H-mean:   54.82 ± 11.97
angry -> depressed
per-class acc:0.6932, seen acc:0.9231, unseen acc:0.4634, H:0.6171
per-class acc:0.6315, seen acc:0.8500, unseen acc:0.4130, H:0.5559
per-class acc:0.7011, seen acc:1.0000, unseen acc:0.4022, H:0.5736
per-class acc:0.8755, seen acc:0.8636, unseen acc:0.8873, H:0.8753
per-class acc:0.5885, seen acc:0.5625, unseen acc:0.6145, H:0.5874
per-class acc:0.5000, seen acc:1.0000, unseen acc:0.0000, H:0.0000
Seen:     86.65 ± 6.62
Unseen:   46.34 ± 11.88
H-mean:   53.49 ± 11.74
angry -> neutral


## Merge results

In [14]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]

df['method'] = pd.Categorical(df['method'], categories=['Base', 'GZSDA', 'Our', 'Our+GRE'], ordered=True)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,angry -> childlike,Base,93.40 ± 2.82,30.35 ± 11.40,38.87 ± 12.93
1,angry -> childlike,GZSDA,90.62 ± 4.19,26.72 ± 11.77,32.82 ± 14.12
2,angry -> childlike,Our,86.81 ± 5.12,46.74 ± 12.14,54.27 ± 11.76
3,angry -> childlike,Our+GRE,85.76 ± 6.49,48.19 ± 12.44,54.82 ± 11.97
4,angry -> depressed,Base,94.08 ± 3.07,15.81 ± 4.59,25.60 ± 6.87
...,...,...,...,...,...
163,strutting -> old,Our+GRE,60.42 ± 8.18,30.41 ± 13.33,30.37 ± 10.54
164,strutting -> proud,Base,75.00 ± 11.18,37.50 ± 13.31,39.63 ± 9.24
165,strutting -> proud,GZSDA,66.67 ± 10.54,28.95 ± 15.51,27.52 ± 12.62
166,strutting -> proud,Our,64.58 ± 9.36,49.56 ± 12.92,46.36 ± 9.71


In [15]:
df.to_csv("./result_actionStyle.csv")

In [16]:
df.groupby("domain")

In [17]:
df["H-mean_value"] = (
    df["unseen"]
    # df["H-mean"]
    .str.split("±")
    .str[0]
    .astype(float)
)
df

,domain,method,seen,unseen,H-mean,H-mean_value
0,angry -> childlike,Base,93.40 ± 2.82,30.35 ± 11.40,38.87 ± 12.93,30.35
1,angry -> childlike,GZSDA,90.62 ± 4.19,26.72 ± 11.77,32.82 ± 14.12,26.72
2,angry -> childlike,Our,86.81 ± 5.12,46.74 ± 12.14,54.27 ± 11.76,46.74
3,angry -> childlike,Our+GRE,85.76 ± 6.49,48.19 ± 12.44,54.82 ± 11.97,48.19
4,angry -> depressed,Base,94.08 ± 3.07,15.81 ± 4.59,25.60 ± 6.87,15.81
...,...,...,...,...,...,...
163,strutting -> old,Our+GRE,60.42 ± 8.18,30.41 ± 13.33,30.37 ± 10.54,30.41
164,strutting -> proud,Base,75.00 ± 11.18,37.50 ± 13.31,39.63 ± 9.24,37.50
165,strutting -> proud,GZSDA,66.67 ± 10.54,28.95 ± 15.51,27.52 ± 12.62,28.95
166,strutting -> proud,Our,64.58 ± 9.36,49.56 ± 12.92,46.36 ± 9.71,49.56


In [18]:
best = df.loc[df.groupby("domain")["H-mean_value"].idxmax()]
best = best[["domain", "method", "H-mean"]].reset_index(drop=True)

best

,domain,method,H-mean
0,angry -> childlike,Our+GRE,54.82 ± 11.97
1,angry -> depressed,Our+GRE,53.49 ± 11.74
2,angry -> neutral,Our+GRE,57.60 ± 12.90
3,angry -> old,Our,54.54 ± 12.16
4,angry -> proud,Our+GRE,58.23 ± 13.11
5,angry -> strutting,Our,56.08 ± 13.22
6,childlike -> angry,Our,49.35 ± 11.72
7,childlike -> depressed,Our+GRE,45.69 ± 12.96
8,childlike -> neutral,Our+GRE,52.13 ± 15.38
9,childlike -> old,Our+GRE,19.79 ± 12.38


In [19]:
from collections import Counter
Counter(best.method)

Counter({'Our+GRE': 21, 'Our': 19, 'Base': 2})